# Exploratory Analysis: Wearable Data And Running Performance

This notebook explores a privacy-safe research dataset derived from the Garmin Connect export and official FIDAL race results. It uses aggregate activity, sleep, HRV, heart-rate proxy, VO2max, training-load, training-status, and race-prediction data. The analysis is exploratory: the dataset is still a single-athlete observational case study, so correlations should be read as hypothesis-generating only.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'

races = pd.read_csv(DATA / 'race_context_dataset.csv', parse_dates=['race_date'])
daily_training = pd.read_csv(DATA / 'daily_training_summary.csv', parse_dates=['date'])
daily_health = pd.read_csv(DATA / 'daily_health_summary.csv', parse_dates=['date'])
races

## Race Context Table

The table below uses FIDAL as official race ground truth for 2023-2026 target events. Garmin auto-detected personal-record values are treated as wearable-derived signals, not official results.

In [ ]:
cols = [
    'race_date', 'fidal_event', 'distance_m', 'official_time_text', 'pace_sec_per_km',
    'run_km_28d', 'sleep_avg_hours_7d', 'hrv_avg_7d', 'rhr_avg_7d',
    'vo2max_nearest', 'acute_load_nearest', 'training_status_nearest',
    'race_prediction_5k_nearest', 'relative_performance_pct'
]
races[cols]

## Running Volume Timeline

In [ ]:
weekly = daily_training.set_index('date').resample('W-MON')['run_km'].sum().reset_index()
weekly.describe()

In [ ]:
from IPython.display import Image, display
for name in [
    'running_volume_timeline.png',
    'race_performance_timeline.png',
    'prerace_28d_running_volume.png',
    'sleep_before_races.png',
    'metric_relationships.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

## Simple Relationships

`relative_performance_pct` is computed relative to the best known official result within the same FIDAL event. Lower is better. These correlations are unstable because the data is from one athlete and mixes event types; they should not be interpreted as predictive evidence.

In [ ]:
relationship_metrics = [
    'run_km_28d', 'sleep_avg_hours_7d', 'hrv_avg_7d', 'hrv_avg_28d',
    'rhr_avg_7d', 'rhr_avg_28d', 'acute_load_nearest', 'vo2max_nearest'
]
corr_rows = []
for metric in relationship_metrics:
    pair = races[[metric, 'relative_performance_pct']].dropna()
    corr_rows.append({
        'metric': metric,
        'n': len(pair),
        'pearson_r_exploratory': pair[metric].corr(pair['relative_performance_pct']) if len(pair) >= 2 else None,
    })
pd.DataFrame(corr_rows)

## Notes For Interpretation

- Pre-race windows exclude race day.
- HRV is available only from 2025-09-18 onward, so older race-context rows would have missing HRV if added.
- The heart-rate field comes from Garmin health status and is treated as a resting/overnight heart-rate proxy, not a lab-confirmed resting heart rate.
- The dataset should be expanded with manually verified race results before any stronger statistical claim is made.